# Program Control Notebook

This notebook is the parent-level operating point for the CE 519 program. All seven modules are now represented here so the full workflow can be executed from one location.

Current active modules:

- Module 1: Steel Reinforced Concrete Pavement Design
- Module 2: Fiber Reinforced Concrete Pavement Design
- Module 3: Life Cycle Costing
- Module 4: Life Cycle Assessment
- Module 5: Uncertainty and Sensitivity Analysis
- Module 6: Optimization and Selection
- Module 7: Summary Output Graphics (Not Yet Implemented)


## Module 1 - Steel Reinforced Concrete Pavement

This cell imports the Module 1 inputs and runner. The default project geometry is already included in `SRCInputs`:

- Parking lot: 300 ft x 150 ft
- Roadway: 2,050 ft x 20 ft
- Total pavement area: 86,000 sf
- Design axle load: 19 kip
- Wheel load: 9.5 kip

Module 1 steel quantities include two-way reinforcement, 40-ft stock rebar lengths, and Class A lap splices. Pavement thickness candidates are limited to even values from 6 to 12 inches.


Reinforcement quantity basis: the parking lot is quantified with two-way reinforcement, while the roadway is quantified with one-way longitudinal reinforcement. Both include 40-ft stock bars and Class A lap splices.


In [ ]:
from pathlib import Path
import pandas as pd

from module_1 import SRCInputs, run_module_1_src
from module_1.src_design import write_results_csv


## Module 1 inputs

Edit these values here when running scenarios. Keeping scenario inputs in this parent notebook avoids burying run-critical values inside the module source code or README.


In [ ]:
inputs = SRCInputs(
    # Project geometry
    parking_lot_length_ft=300.0,
    parking_lot_width_ft=150.0,
    roadway_length_ft=2050.0,
    roadway_width_ft=20.0,

    # Loading
    axle_load_kip=19.0,
    tire_pressure_psi=100.0,

    # Materials and support
    concrete_fpc_psi=4500.0,
    concrete_MR_psi=650.0,
    subgrade_k_pci=100.0,

    # Reinforcement placement
    clear_cover_bottom_in=3.0,
)

# Module 1 quantities include two-way reinforcement with 40-ft stock bars and Class A lap splices.
reinforcement_directions = 2

print(f'Total CE 519 pavement area = {inputs.total_area_sf:,.0f} sf')
print(f'Design wheel load = {inputs.wheel_load_lb / 1000:.2f} kip')


## Run Module 1

This evaluates all discrete Module 1 alternatives and separates feasible results from the full candidate list.


In [ ]:
all_results, feasible_results = run_module_1_src(
    inputs,
    reinforcement_directions=reinforcement_directions,
)

print(f'Total candidates checked: {len(all_results):,}')
print(f'Feasible candidates: {len(feasible_results):,}')


## Save Module 1 outputs

The output CSV files are written to the parent-level `outputs` folder. Later modules can read these files or use the in-memory DataFrames directly.


In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

write_results_csv(all_results, output_dir / 'module_1_all_results.csv')
write_results_csv(feasible_results, output_dir / 'module_1_feasible_results.csv')

all_df = pd.DataFrame(all_results)
feasible_df = pd.DataFrame(feasible_results)

print('Saved:')
print(output_dir / 'module_1_all_results.csv')
print(output_dir / 'module_1_feasible_results.csv')


## Review feasible Module 1 results

The table below sorts feasible alternatives by demand/capacity ratio and concrete volume. This is only an initial engineering review view; final ranking should occur in the later selection/optimization module.


In [ ]:
review_columns = [
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'demand_capacity_ratio',
    'cracking_ratio',
    'concrete_volume_cy_total',
    'subbase_volume_cy_total',
    'steel_weight_ton_total',
]

feasible_df[review_columns].sort_values(
    by=['pavement_thickness_in', 'steel_weight_ton_total', 'demand_capacity_ratio']
).head(20)


## Module 2 - Fiber Reinforced Concrete Pavement

Module 2 uses the same project geometry, loading, concrete properties, and support assumptions as Module 1 where appropriate. The FRC decision variable is `fe3`, the equivalent/residual flexural strength input.

Pavement thickness candidates are limited to even values from 6 to 12 inches.


In [ ]:
from module_2 import FRCInputs, generate_frc_candidates, run_module_2_frc
from module_2.frc_design import write_results_csv as write_frc_results_csv


## Module 2 inputs

These inputs mirror the appropriate Module 1 inputs. Update `fe3_values_psi` when project-specific FRC performance values are selected.


In [ ]:
frc_inputs = FRCInputs(
    # Project geometry
    parking_lot_length_ft=inputs.parking_lot_length_ft,
    parking_lot_width_ft=inputs.parking_lot_width_ft,
    roadway_length_ft=inputs.roadway_length_ft,
    roadway_width_ft=inputs.roadway_width_ft,

    # Loading
    axle_load_kip=inputs.axle_load_kip,
    tire_pressure_psi=inputs.tire_pressure_psi,

    # Materials and support
    concrete_fpc_psi=inputs.concrete_fpc_psi,
    concrete_MR_psi=inputs.concrete_MR_psi,
    concrete_Ec_psi=inputs.concrete_Ec_psi,
    poisson_ratio=inputs.poisson_ratio,
    subgrade_k_pci=inputs.subgrade_k_pci,
    phi_frc=1.0,
)

# Adjust these if project-specific FRC performance values are updated.
fe3_values_psi = [100, 150, 200, 250, 300, 350, 400]

frc_candidates = generate_frc_candidates(
    fe3_values_psi=fe3_values_psi,
)

print(f'Total CE 519 pavement area = {frc_inputs.total_area_sf:,.0f} sf')
print(f'Design wheel load = {frc_inputs.wheel_load_lb / 1000:.2f} kip')
print(f'FRC fe3 values checked = {fe3_values_psi}')


## Run Module 2

This evaluates all discrete Module 2 alternatives and separates feasible results from the full candidate list.


In [ ]:
module_2_all_results, module_2_feasible_results = run_module_2_frc(
    frc_inputs,
    candidates=frc_candidates,
)

print(f'Total FRC candidates checked: {len(module_2_all_results):,}')
print(f'Feasible FRC candidates: {len(module_2_feasible_results):,}')


## Save Module 2 outputs

The output CSV files are written to the parent-level `outputs` folder. Later modules can read these files or use the in-memory DataFrames directly.


In [ ]:
write_frc_results_csv(module_2_all_results, output_dir / 'module_2_all_results.csv')
write_frc_results_csv(module_2_feasible_results, output_dir / 'module_2_feasible_results.csv')

module_2_all_df = pd.DataFrame(module_2_all_results)
module_2_feasible_df = pd.DataFrame(module_2_feasible_results)

print('Saved:')
print(output_dir / 'module_2_all_results.csv')
print(output_dir / 'module_2_feasible_results.csv')


## Review feasible Module 2 results

This view sorts feasible FRC alternatives by pavement thickness, fe3, and demand/capacity ratio.


In [ ]:
frc_review_columns = [
    'subbase_thickness_in',
    'pavement_thickness_in',
    'fe3_psi',
    'Re3_percent',
    'Mu_kip_in_per_ft',
    'Mn_FRC_kip_in_per_ft',
    'Mtotal_FRC_kip_in_per_ft',
    'phi_Mtotal_FRC_kip_in_per_ft',
    'demand_capacity_ratio',
    'cracking_ratio',
    'concrete_volume_cy_total',
    'subbase_volume_cy_total',
]

module_2_feasible_df[frc_review_columns].sort_values(
    by=['pavement_thickness_in', 'fe3_psi', 'demand_capacity_ratio']
).head(20)


## Module 3 - Life Cycle Costing

Module 3 consumes the feasible Module 1 and Module 2 alternatives and calculates present-worth life-cycle cost. For this project, maintenance is set to zero, and end-of-life includes full demolition for reuse as crushed concrete aggregate.


In [ ]:
from module_3 import LCCInputs, RSMeansUnitCosts, run_module_3_lcc
from module_3.lcc_design import write_results_csv as write_lcc_results_csv


## Module 3 cost inputs

Update these values using the RSMeans line items selected for the CE 519 program. Enter national-average RSMeans values first; the Saginaw County factor is applied separately.


In [ ]:
rsmeans_costs = RSMeansUnitCosts(
    # Adjust these baseline values with the selected RSMeans line items.
    concrete_cost_per_cy=185.00,
    stone_57_cost_per_cy=58.00,
    reinforcing_steel_cost_per_ton=3200.00,

    # Baseline FRC fiber cost uses the maximum documented contract value.
    # Module 5 samples $1.323 to $1.47/lb as potential volume-discount savings.
    tufstrand_sf_cost_per_lb=1.47,

    # End-of-service-life costs.
    concrete_demolition_cost_per_cy=42.00,
    concrete_crushing_cost_per_ton=9.00,
    subbase_removal_cost_per_cy=18.00,
    recycled_concrete_aggregate_credit_per_ton=6.00,
)

lcc_inputs = LCCInputs(
    analysis_period_yr=50,
    end_of_service_life_yr=50,
    real_discount_rate=0.03,

    # Enter the current RSMeans / CCI factor for Saginaw County or nearest listed city.
    saginaw_county_location_factor=1.00,

    # No maintenance by project basis.
    maintenance_cost_present_worth=0.0,
)


## Run Module 3

This uses the in-memory feasible results from Modules 1 and 2. If those cells have not been run, run Modules 1 and 2 first.


In [ ]:
module_3_lcc_results = run_module_3_lcc(
    module_1_results=feasible_results,
    module_2_results=module_2_feasible_results,
    costs=rsmeans_costs,
    inputs=lcc_inputs,
)

module_3_lcc_df = pd.DataFrame(module_3_lcc_results)

print(f'LCC alternatives evaluated: {len(module_3_lcc_results):,}')


## Save Module 3 outputs


In [ ]:
write_lcc_results_csv(module_3_lcc_results, output_dir / 'module_3_lcc_results.csv')

print('Saved:')
print(output_dir / 'module_3_lcc_results.csv')


## Review lowest present-worth alternatives


In [ ]:
lcc_review_columns = [
    'module',
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'fe3_psi',
    'initial_construction_cost',
    'end_of_life_present_worth',
    'total_present_worth',
    'present_worth_per_sf',
]

available_lcc_columns = [c for c in lcc_review_columns if c in module_3_lcc_df.columns]

module_3_lcc_df[available_lcc_columns].sort_values(
    by=['total_present_worth']
).head(20)


## Module 4 - Life Cycle Assessment

Module 4 calculates TRACI-style environmental outputs for feasible Module 1 and Module 2 alternatives. Climate change remains the primary selection metric, with additional screening categories retained for summary graphics.

In [ ]:
from module_4 import LCAInputs, LCAUnitImpacts, FacilityLocation, run_module_4_lca
from module_4.lca_design import (
    geocode_address_nominatim,
    route_distance_miles_osrm,
    write_results_csv as write_lca_results_csv,
)


## Module 4 locations and haul distances

Short names are used below for the CE 519 haul-distance inputs. The OpenStreetMap/OSRM cell can be run when internet access is available. If it is not run, the default distances in `LCAInputs` are used.


In [ ]:
project_site = FacilityLocation(
    short_name='Project site',
    address='561 N Hemlock Rd, Hemlock, MI 48626',
)

stone_source = FacilityLocation(
    short_name='Wirt Stone Dock',
    address='Wirt Stone Dock, Saginaw, MI',
    latitude=43.4738915,
    longitude=-83.9080425,
)

concrete_source = FacilityLocation(
    short_name='R & R Ready Mix',
    address='R & R Ready Mix Inc, Saginaw, MI',
    latitude=43.4114697,
    longitude=-84.2327798,
)

print(stone_source)
rebar_source = FacilityLocation(
    short_name='HYMMCO',
    address='HYMMCO, Saginaw, MI',
    latitude=43.5037881,
    longitude=-83.9727579,
)

nucor_source = FacilityLocation(
    short_name='Nucor Marion OH',
    address='Nucor Marion, OH',
    latitude=40.5887,
    longitude=-83.1285,
)

print(concrete_source)
print(nucor_source)
print(rebar_source)
print(project_site)


### Calculate route distances using OpenStreetMap

Run this cell with internet access to calculate one-way route distances. The #57 stone haul distance and crushed concrete haul distance use the same Wirt Stone Dock route.


In [ ]:
# Online distance calculation.
# If this fails because internet access is unavailable, manually enter distances in the next cell.

try:
    project_lat, project_lon = geocode_address_nominatim(project_site.address)

    stone_to_project_miles = route_distance_miles_osrm(
        stone_source.latitude,
        stone_source.longitude,
        project_lat,
        project_lon,
    )

    concrete_to_project_miles = route_distance_miles_osrm(
        concrete_source.latitude,
        concrete_source.longitude,
        project_lat,
        project_lon,
    )

    crushed_concrete_to_reuse_miles = stone_to_project_miles
    rebar_to_project_miles = 12.0
    nucor_to_hymmco_miles = 195.0

    rebar_to_project_miles = route_distance_miles_osrm(
        rebar_source.latitude,
        rebar_source.longitude,
        project_lat,
        project_lon,
    )

    nucor_to_hymmco_miles = route_distance_miles_osrm(
        nucor_source.latitude,
        nucor_source.longitude,
        rebar_source.latitude,
        rebar_source.longitude,
    )

    print(f'Project coordinates: {project_lat:.6f}, {project_lon:.6f}')
    print(f'Wirt Stone Dock to project: {stone_to_project_miles:.2f} miles')
    print(f'R & R Ready Mix to project: {concrete_to_project_miles:.2f} miles')
    print(f'Crushed concrete to Wirt Stone Dock: {crushed_concrete_to_reuse_miles:.2f} miles')
    print(f'Nucor Marion OH to HYMMCO: {nucor_to_hymmco_miles:.2f} miles')
    print(f'HYMMCO rebar to project: {rebar_to_project_miles:.2f} miles')

except Exception as exc:
    print('Online distance calculation did not complete.')
    print(exc)
    stone_to_project_miles = 16.0
    concrete_to_project_miles = 18.0
    crushed_concrete_to_reuse_miles = stone_to_project_miles
    rebar_to_project_miles = 12.0
    nucor_to_hymmco_miles = 195.0


## Module 4 LCA inputs

Update the unit impact factors once the final ecoinvent APOS AO / SimaPro factors are selected. Transport is tracked as ton-miles and metric tonne-km so the inventory can be checked against ecoinvent lorry process units.

In [ ]:
lca_unit_impacts = LCAUnitImpacts(
    # Replace ecoinvent factors with final APOS AO/SimaPro results when available.
    concrete_kgco2e_per_cy=355.0,
    stone_57_kgco2e_per_ton=5.0,
    # CRSI fabricated rebar EPD A1-A3: 854 kg CO2-eq/metric ton = 774.736 kg CO2-eq/short ton.
    reinforcing_steel_kgco2e_per_ton=774.736,

    # Euclid TUF-STRAND SF TDS: 3.08 kg CO2-eq/kg.
    tufstrand_sf_kgco2e_per_kg=3.08,

    truck_transport_kgco2e_per_ton_mile=0.17,
    demolition_kgco2e_per_cy_concrete=3.0,
    concrete_crushing_kgco2e_per_ton=1.5,
    virgin_aggregate_credit_kgco2e_per_ton=5.0,
)

# TUF-STRAND dosage is calculated in Module 4 as:
# dosage, lb/yd3 = 0.03*fe3 - 1.1, limited to 3 to 20 lb/yd3.
lca_inputs = LCAInputs(
    stone_to_project_miles=stone_to_project_miles,
    concrete_to_project_miles=concrete_to_project_miles,
    crushed_concrete_to_reuse_miles=crushed_concrete_to_reuse_miles,
    rebar_to_project_miles=rebar_to_project_miles,
    nucor_to_hymmco_miles=nucor_to_hymmco_miles,
    include_virgin_aggregate_credit=False,
)


## Run Module 4

This uses the in-memory feasible results from Modules 1 and 2. If those cells have not been run, run Modules 1 and 2 first.


In [ ]:
module_4_lca_results = run_module_4_lca(
    module_1_results=feasible_results,
    module_2_results=module_2_feasible_results,
    unit_impacts=lca_unit_impacts,
    inputs=lca_inputs,
)

module_4_lca_df = pd.DataFrame(module_4_lca_results)

print(f'LCA alternatives evaluated: {len(module_4_lca_results):,}')


## Save Module 4 outputs


In [ ]:
write_lca_results_csv(module_4_lca_results, output_dir / 'module_4_lca_results.csv')

print('Saved:')
print(output_dir / 'module_4_lca_results.csv')


## Review lowest GWP alternatives


In [ ]:
lca_review_columns = [
    'module',
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'fe3_psi',
    'gwp_concrete_kgco2e',
    'gwp_57_stone_kgco2e',
    'gwp_reinforcing_steel_kgco2e',
    'gwp_tufstrand_sf_kgco2e',
    'gwp_transport_kgco2e',
    'haul_rebar_ton_miles',
    'haul_rebar_nucor_to_hymmco_ton_miles',
    'haul_rebar_hymmco_to_project_ton_miles',
    'gwp_demolition_kgco2e',
    'gwp_crushing_kgco2e',
    'gwp_total_project_kgco2e',
]

available_lca_columns = [c for c in lca_review_columns if c in module_4_lca_df.columns]

module_4_lca_df[available_lca_columns].sort_values(
    by=['gwp_total_project_kgco2e']
).head(20)


## Module 5 - Uncertainty and Sensitivity Analysis

Module 5 runs Monte Carlo uncertainty analysis and Spearman rank sensitivity. Cost, LCA factor, and subgrade modulus `k` uncertainty are included; `k` is used only for design-demand sensitivity screening at this stage.

In [ ]:
from module_5 import (
    UncertaintyInputs,
    build_default_uncertain_parameters,
    run_module_5_uncertainty,
)
from module_5.uncertainty_design import write_results_csv as write_uncertainty_results_csv


## Module 5 setup

The default run uses 50,000 simulations and random seed 42. For quick debugging, temporarily reduce `n_simulations`. The current design uncertainty variable is limited to subgrade modulus `k`.

In [ ]:
uncertainty_inputs = UncertaintyInputs(
    n_simulations=50_000,
    random_seed=42,
    end_of_service_life_yr=lcc_inputs.end_of_service_life_yr,
    base_subgrade_k_pci=inputs.subgrade_k_pci,
    concrete_ton_per_cy=lcc_inputs.concrete_ton_per_cy,
    stone_ton_per_cy=lca_inputs.stone_ton_per_cy,
)

uncertain_parameters = build_default_uncertain_parameters()

pd.DataFrame([p.__dict__ for p in uncertain_parameters])


## Run Module 5

This cell uses the LCC and LCA results from Modules 3 and 4. Run Modules 1 through 4 first.


In [ ]:
(
    module_5_parameter_table,
    module_5_uncertainty_summary,
    module_5_spearman_sensitivity,
) = run_module_5_uncertainty(
    lcc_results=module_3_lcc_results,
    lca_results=module_4_lca_results,
    parameters=uncertain_parameters,
    inputs=uncertainty_inputs,
)

module_5_parameter_df = pd.DataFrame(module_5_parameter_table)
module_5_summary_df = pd.DataFrame(module_5_uncertainty_summary)
module_5_sensitivity_df = pd.DataFrame(module_5_spearman_sensitivity)

print(f'Uncertainty alternatives evaluated: {len(module_5_summary_df):,}')
print(f'Sensitivity rows: {len(module_5_sensitivity_df):,}')


## Save Module 5 outputs


In [ ]:
write_uncertainty_results_csv(
    module_5_parameter_table,
    output_dir / 'module_5_uncertainty_parameter_table.csv',
)

write_uncertainty_results_csv(
    module_5_uncertainty_summary,
    output_dir / 'module_5_uncertainty_summary.csv',
)

write_uncertainty_results_csv(
    module_5_spearman_sensitivity,
    output_dir / 'module_5_spearman_sensitivity.csv',
)

print('Saved Module 5 outputs to the outputs folder.')


## Review Module 5 uncertainty summary


In [ ]:
module_5_summary_df.sort_values(
    by=['total_present_worth_mean']
).head(20)


## Review strongest sensitivity results


In [ ]:
module_5_sensitivity_df.sort_values(
    by=['abs_spearman_rho'],
    ascending=False,
).head(30)


## Module 6 - Optimization and Selection

Module 6 applies the final selection rule: lowest LCA controls as long as the alternative is within 120% of the lowest deterministic LCC solution. Module 5 uncertainty and sensitivity results are attached only for the reported solutions.


In [ ]:
from module_6 import SelectionInputs, run_module_6_selection
from module_6.selection_design import write_results_csv as write_selection_results_csv


## Run Module 6

This cell uses deterministic Module 3 LCC and Module 4 LCA for selection. Module 5 outputs are used only to report uncertainty and sensitivity for the selected solutions.


In [ ]:
selection_inputs = SelectionInputs(
    cost_threshold_multiplier=1.20,
)

(
    module_6_benchmark_solution,
    module_6_selected_solutions,
    module_6_eligible_solutions,
    module_6_selected_sensitivity,
) = run_module_6_selection(
    lcc_results=module_3_lcc_results,
    lca_results=module_4_lca_results,
    uncertainty_summary=module_5_uncertainty_summary,
    sensitivity_results=module_5_spearman_sensitivity,
    inputs=selection_inputs,
)

module_6_benchmark_df = pd.DataFrame(module_6_benchmark_solution)
module_6_selected_df = pd.DataFrame(module_6_selected_solutions)
module_6_eligible_df = pd.DataFrame(module_6_eligible_solutions)
module_6_selected_sensitivity_df = pd.DataFrame(module_6_selected_sensitivity)

print(f'Eligible alternatives within 120% LCC threshold: {len(module_6_eligible_df):,}')
print(f'Reported selected/benchmark alternatives: {len(module_6_selected_df):,}')


## Save Module 6 outputs


In [ ]:
write_selection_results_csv(
    module_6_benchmark_solution,
    output_dir / 'module_6_benchmark_lowest_lcc_solution.csv',
)

write_selection_results_csv(
    module_6_selected_solutions,
    output_dir / 'module_6_selected_solutions.csv',
)

write_selection_results_csv(
    module_6_eligible_solutions,
    output_dir / 'module_6_eligible_solutions.csv',
)

write_selection_results_csv(
    module_6_selected_sensitivity,
    output_dir / 'module_6_selected_solution_sensitivity.csv',
)

print('Saved Module 6 outputs to the outputs folder.')


## Review selected solutions


In [ ]:
selection_review_columns = [
    'selection_role',
    'selection_status',
    'module',
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'fe3_psi',
    'total_present_worth',
    'lcc_percent_of_benchmark',
    'gwp_total_project_kgco2e',
    'total_present_worth_p05',
    'total_present_worth_p50',
    'total_present_worth_p95',
    'gwp_total_project_kgco2e_p05',
    'gwp_total_project_kgco2e_p50',
    'gwp_total_project_kgco2e_p95',
]

available_selection_columns = [c for c in selection_review_columns if c in module_6_selected_df.columns]
module_6_selected_df[available_selection_columns]


## Review selected-solution sensitivities


In [ ]:
module_6_selected_sensitivity_df.sort_values(
    by=['alternative_id', 'output', 'sensitivity_rank']
).groupby(['alternative_id', 'output']).head(5)


## Module 7 - Summary Output Graphics

Module 7 generates presentation/report graphics from the CSV outputs produced by Modules 1 through 6. Figures are written to `outputs/figures/`, and a manifest is written to `outputs/module_7_figure_manifest.csv`.

In [ ]:
from module_7 import SummaryOutputInputs, run_module_7_summary_output

summary_output_inputs = SummaryOutputInputs(
    output_dir=output_dir,
    figure_dir_name='figures',
    dpi=300,
)

## Run Module 7

In [ ]:
module_7_figure_manifest = run_module_7_summary_output(summary_output_inputs)
module_7_figure_manifest

## Review Module 7 figure manifest

In [ ]:
module_7_manifest_df = pd.DataFrame(module_7_figure_manifest)
module_7_manifest_df